[//]: # (cr:doc name='chapter_9_business_alignment' id=1e6802b8)
# Chapter 9: Business Alignment

**Purpose:** Align data exploration with business objectives and constraints.

**Outputs:**
- Business context documentation
- Success metrics definition
- Constraints and requirements

---

[//]: # (cr:doc name='9_1_setup' id=291677a7)
## 9.1 Setup

In [ ]:
# @cr:code name='init_progress' id=7333872d
from IPython.display import Markdown, display

from customer_retention.analysis.notebook_progress import accept_workflow_params, track_and_export_previous
from customer_retention.analysis.visualization import display_table

accept_workflow_params()
track_and_export_previous("09_business_alignment.ipynb")


from customer_retention.analysis.auto_explorer import ExplorationFindings
from customer_retention.core.compat import native_pd
from customer_retention.core.config.experiments import (
    FINDINGS_DIR,
)

# --- cr:profiler ---
if __import__('os').environ.get("CR_BATCH_EXECUTION") == "1":
    import json as _j
    import os as _os
    import re as _r
    _cr_nb = _os.path.splitext(_os.path.basename(_os.environ.get("PAPERMILL_OUTPUT_PATH", "")))[0]
    if _cr_nb:
        _cr_mp = _os.path.join(_os.getcwd(), f".cr_cell_metrics_{_cr_nb}.jsonl")
        open(_cr_mp, 'w').close()
        _cr_re = _r.compile(r"^#\s*@cr:\w+\s+name='([^']+)'\s+id=(\w+)")
        def _cr_jc():
            return -1
        try:
            _s = __import__('pyspark.sql', fromlist=['SparkSession']).SparkSession.getActiveSession()
            if _s:
                def _cr_jc():  # noqa: F811
                    return _s._jsc.sc().dagScheduler().nextJobId().get()
        except Exception:
            pass
        def _cr_pre(info):
            info._cr_sj = _cr_jc()
        def _cr_post(r):
            sj = getattr(r.info, '_cr_sj', -1)
            sa = _cr_jc()
            m = _cr_re.match((r.info.raw_cell or '').split('\n')[0])
            if m:
                with open(_cr_mp, 'a') as f:
                    f.write(_j.dumps({"cell_name": m.group(1), "cell_id": m.group(2),
                                      "spark_jobs": (sa - sj) if sj >= 0 and sa >= 0 else None}) + '\n')
        get_ipython().events.register('pre_run_cell', _cr_pre)
        get_ipython().events.register('post_run_cell', _cr_post)
# --- /cr:profiler ---


In [ ]:
# @cr:code name='load_findings' id=35b16e01
from customer_retention.analysis.auto_explorer import load_notebook_findings

FINDINGS_PATH, _namespace, _ = load_notebook_findings("09_business_alignment.ipynb")
print(f"Using: {FINDINGS_PATH}")

findings = ExplorationFindings.load(FINDINGS_PATH)
print(f"\nLoaded findings for {findings.column_count} columns")

[//]: # (cr:doc name='9_model_diagnostics' id=32ea2cae)

## Model Diagnostics

Assess model health before business alignment: CV stability, feature consistency, overfitting, leakage, calibration.

In [ ]:
# @cr:code name='load_diagnostics_inputs' id=0cf10f74
import json as _json

_skip_diagnostics = True
_diag_data = None
_diag_report = None
_holdout_metrics = None

if _namespace and _namespace.exploration_diagnostics_path.exists():
    _diag_data = _json.loads(_namespace.exploration_diagnostics_path.read_text())
    _diag_report = _diag_data.get("model_diagnostics_report")
    _holdout_metrics = _diag_data.get("best_model_holdout_metrics")
    _skip_diagnostics = _diag_report is None
    if _skip_diagnostics:
        print("exploration_diagnostics.json present but missing model_diagnostics_report — re-run NB08")
    else:
        _summary_count = len((_diag_report or {}).get("summaries", {}))
        print(f"Loaded persisted diagnostics for {_summary_count} model(s)")
        if _holdout_metrics:
            print(f"Holdout metrics available for best model: {_holdout_metrics.get('model_name')}")
else:
    print("No exploration_diagnostics.json found (run NB08 first)")


In [ ]:
# @cr:code name='render_diagnostics_summary' id=a2809a2a
if not _skip_diagnostics and _diag_report:
    _verdict = (_diag_report.get("verdict") or "unknown").upper()
    print(f"Diagnostics verdict (computed in NB08): {_verdict}")


In [ ]:
# @cr:code name='display_cv_consistency' id=5ef7b8f4
if not _skip_diagnostics and _diag_report:
    from customer_retention.analysis.visualization import display_table

    _cv_rows = []
    for _name, _s in _diag_report.get("summaries", {}).items():
        _cv = _s.get("cv_analysis") or {}
        _outliers = _cv.get("outlier_folds")
        _cv_rows.append({
            "Model": _name,
            "CV Mean": f"{(_cv.get('cv_mean') or 0.0):.4f}",
            "CV Std": f"{(_cv.get('cv_std') or 0.0):.4f}",
            "Stable": "Yes" if _cv.get("passed") else "No",
            "Best-Worst Gap": f"{(_cv.get('best_worst_gap') or 0.0):.4f}",
            "Outlier Folds": str(_outliers) if _outliers else "-",
        })
    display(Markdown("### Cross-Validation Consistency"))
    display_table(native_pd.DataFrame(_cv_rows))


In [ ]:
# @cr:code name='display_feature_stability' id=bada3fed
if not _skip_diagnostics and _diag_report:
    display(Markdown("### Feature Importance Stability Across Folds"))
    for _name, _s in _diag_report.get("summaries", {}).items():
        _fs = _s.get("feature_stability")
        if _fs:
            print(f"\n{_name}: overall stability = {(_fs.get('overall_stability') or 0.0):.2f}")
            _stable = _fs.get("stable_features") or []
            print(f"  Stable features ({len(_stable)}): {', '.join(_stable[:10])}")
            _vol = _fs.get("volatile_features") or []
            if _vol:
                print(f"  Volatile features ({len(_vol)}): {', '.join(_vol[:10])}")
        else:
            print(f"\n{_name}: no fold-level importances available")

    _agreement = _diag_report.get("cross_model_agreement") or {}
    display(Markdown("### Cross-Model Feature Agreement"))
    print(f"Overall agreement score: {(_agreement.get('agreement_score') or 0.0):.2f}")
    _consensus = _agreement.get("consensus_features") or []
    if _consensus:
        print(f"Consensus features (all models agree): {', '.join(_consensus[:15])}")
    for _pair, _jac in (_agreement.get("pairwise_jaccard") or {}).items():
        print(f"  {_pair}: Jaccard = {_jac:.2f}")


In [ ]:
# @cr:code name='display_leakage_and_overfitting' id=355a258b
if not _skip_diagnostics and _diag_report:
    display(Markdown("### Leakage & Overfitting Analysis"))

    _leakage = _diag_report.get("leakage") or {}
    _critical_leaks = [
        c for c in (_leakage.get("checks") or [])
        if ((c.get("severity") or {}).get("name") in ("CRITICAL", "HIGH"))
    ]
    if _critical_leaks:
        print(f"  LEAKAGE WARNINGS ({len(_critical_leaks)})")
        for _c in _critical_leaks[:5]:
            _sev = (_c.get("severity") or {}).get("name", "?")
            print(f"    [{_sev}] {_c.get('check_id', '?')}: {_c.get('feature', '?')} — {_c.get('recommendation', '')}")
    else:
        print("  No critical leakage detected")

    print()
    for _name, _s in _diag_report.get("summaries", {}).items():
        _gap_checks = [c for c in ((_s.get("overfitting") or {}).get("checks") or []) if c.get("gap") is not None]
        if _gap_checks:
            for _c in _gap_checks:
                _sev = (_c.get("severity") or {}).get("name", "?")
                print(f"  {_name}: {_c.get('metric', '?')} gap={_c.get('gap', 0.0):.4f} [{_sev}]")
        else:
            print(f"  {_name}: no significant train-test gap")

    _lc_block = _diag_report.get("best_model_learning_curve")
    if _lc_block:
        _lc = _lc_block.get("learning_curve") or []
        if _lc:
            display(Markdown(f"### Learning Curve ({_diag_data.get('best_model_name', '?')})"))
            display_table(native_pd.DataFrame(_lc))


In [ ]:
# @cr:code name='display_calibration_and_verdict' id=7014d9fe
if not _skip_diagnostics and _diag_report:
    display(Markdown("### Calibration"))
    for _name, _s in _diag_report.get("summaries", {}).items():
        _cal = _s.get("calibration") or {}
        print(
            f"  {_name}: Brier={(_cal.get('brier_score') or 0.0):.4f}  "
            f"ECE={(_cal.get('ece') or 0.0):.4f}  MCE={(_cal.get('mce') or 0.0):.4f}  \u2192 {_cal.get('recommendation', '')}"
        )

    _coverage = _diag_report.get("leakage_coverage") or {}
    if _coverage:
        display(Markdown("### Leakage Coverage (NB05 cache)"))
        _total = _coverage.get("total_features", 0)
        _analyzed = _coverage.get("analyzed_in_nb05", 0)
        print(f"  {_analyzed}/{_total} features have NB05 cached correlations")
        _unanalyzed = _coverage.get("unanalyzed") or []
        if _unanalyzed:
            _missing = _total - _analyzed
            print(f"  {_missing} feature(s) added after NB05 (gold transforms / silver derived) — not in cached leakage view")
            print(f"  Sample: {', '.join(_unanalyzed)}")

    _skipped = _diag_report.get("skipped_analyses") or []
    if _skipped:
        display(Markdown("### Skipped Analyses"))
        for _entry in _skipped:
            print(f"  \u2022 {_entry}")

    display(Markdown("---"))
    _v = (_diag_report.get("verdict") or "unknown").upper()
    _emoji = {"SOLID": "PASS", "CAUTION": "REVIEW", "OVERFIT": "FAIL", "LEAKY": "FAIL", "UNSTABLE": "FAIL"}
    display(Markdown(f"## Model Verdict: **{_v}** ({_emoji.get(_v, '?')})"))

    for _issue in _diag_report.get("critical_issues") or []:
        print(f"  - {_issue}")
    if _diag_report.get("recommendations"):
        display(Markdown("### Recommendations"))
        for _rec in _diag_report["recommendations"]:
            print(f"  - {_rec}")

In [ ]:
# @cr:code name='display_best_exploration_holdout' id=c3a7f1e2
if not _skip_diagnostics and _holdout_metrics:
    _best_name = _holdout_metrics.get("model_name", "?")
    display(Markdown(f"### Best Exploration Model: {_best_name}"))
    print(f"  ROC-AUC:   {_holdout_metrics.get('roc_auc', float('nan')):.4f}")
    print(f"  PR-AUC:    {_holdout_metrics.get('pr_auc', float('nan')):.4f}")
    print(f"  F1:        {_holdout_metrics.get('f1', float('nan')):.4f}")
    print(f"  Precision: {_holdout_metrics.get('precision', float('nan')):.4f}")
    print(f"  Recall:    {_holdout_metrics.get('recall', float('nan')):.4f}")
    print(f"  Accuracy:  {_holdout_metrics.get('accuracy', float('nan')):.4f}")
    _cm = _holdout_metrics.get("confusion_matrix") or {}
    print(f"\n  Confusion Matrix:  TN={_cm.get('tn', 0):,}  FP={_cm.get('fp', 0):,}")
    print(f"                     FN={_cm.get('fn', 0):,}  TP={_cm.get('tp', 0):,}")
elif not _skip_diagnostics:
    print("Best-model holdout metrics not present in exploration_diagnostics.json — re-run NB08")


[//]: # (cr:doc name='9_feature_provenance' id=fa90b1d0)

### Feature Provenance — origin of top features and NB05 cross-check

For each baseline model, walks the top-importance features back to the
raw column and source dataset they were derived from, and joins each row
against NB05's cached target correlations. Three statuses surface here:

- **confirmed** — NB05 saw this feature (or its base column) and cached a
  non-trivial target correlation. The model is leaning on signal NB05
  already vetted.
- **new** — NB05 never saw this feature. It was added by gold transforms
  (`_is_zero`, `_log`) or silver derivations after NB05 ran. **High count
  here is the same blind spot the run-451071b6 evaluation flagged** —
  297/300 of the trained features were post-NB05. With the LD062/LD063
  gold-output gate now active these still surface in the leakage section
  above; this view tells you which sources they cluster in.
- **missing** — NB05 saw the feature but cached a near-zero correlation.
  The model is using something NB05 didn't think mattered.

The `original_dtype` column shows the inferred type of the raw column the
derived feature was built from (numeric, categorical, datetime, …). It is
blank for interaction / ratio features because they combine multiple
inputs — a single dtype would be misleading.

The histogram below each per-model table shows how that model's top-50
features split across source datasets. The overall pie chart at the
bottom counts **every unique feature** in any model's trained feature set
and groups by source, giving a single view of which datasets the trained
models lean on the most. A healthy run distributes features across
multiple sources; a single source dominating is the fingerprint of a
redundant leakage path.


In [ ]:
# @cr:code name='display_feature_provenance' id=fa90b1c0
if not _skip_diagnostics and _diag_report and _namespace:
    import plotly.express as _px

    from customer_retention.analysis.auto_explorer.layered_recommendations import RecommendationRegistry
    from customer_retention.analysis.diagnostics.feature_provenance import (
        build_overall_source_distribution,
        build_provenance_table,
        build_source_column_map,
        build_source_column_types,
        cached_target_correlations,
        source_histogram,
    )

    _source_columns = build_source_column_map(_namespace.multi_dataset_findings_path)
    _source_column_types = build_source_column_types(_namespace.multi_dataset_findings_path)
    _cached_corrs: dict = {}
    if _namespace.merged_recommendations_path.exists():
        try:
            _recs_for_prov = RecommendationRegistry.load(_namespace.merged_recommendations_path)
            _cached_corrs = cached_target_correlations(_recs_for_prov)
        except Exception as _exc:
            print(f'  [provenance] could not load cached correlations: {_exc}')

    _model_importances: dict = {}
    display(Markdown('### Top Feature Provenance Per Model'))
    for _model_name, _summary in _diag_report.get('summaries', {}).items():
        _stability = _summary.get('feature_stability') or {}
        _stats = _stability.get('importance_stats') or {}
        _imps = {feat: float((s or {}).get('mean', 0.0)) for feat, s in _stats.items()}
        if not _imps:
            print(f'  {_model_name}: no per-fold importance stats available')
            continue
        _model_importances[_model_name] = _imps
        _rows = build_provenance_table(
            _imps, _source_columns, _cached_corrs,
            top_n=50, source_column_types=_source_column_types,
        )
        display(Markdown(f'#### {_model_name}'))
        _table = native_pd.DataFrame([{
            'feature': r.feature,
            'source': r.source,
            'original_dtype': r.original_dtype or '',
            'base_column': r.base_column,
            'family': r.family,
            'lag': r.lag_prefix,
            'importance': round(r.importance, 4),
            'NB05_corr': (round(r.nb05_correlation, 3) if r.nb05_correlation is not None else None),
            'status': r.nb05_status,
        } for r in _rows])
        display_table(_table)

        _hist = source_histogram(_rows)
        if _hist:
            _hist_table = native_pd.DataFrame([
                {'source': s, 'top_feature_count': c, 'mean_importance': round(m, 4)}
                for s, c, m in _hist
            ])
            display(Markdown(f'**{_model_name} — top features by source dataset**'))
            display_table(_hist_table)

        _statuses = {'confirmed': 0, 'new': 0, 'missing': 0}
        for r in _rows:
            _statuses[r.nb05_status] = _statuses.get(r.nb05_status, 0) + 1
        _total = max(sum(_statuses.values()), 1)
        print(
            f'  NB05 coverage of top-{len(_rows)}: '
            f"confirmed={_statuses['confirmed']} "
            f"new={_statuses['new']} "
            f"missing={_statuses['missing']}  "
            f"({100 * _statuses['confirmed'] // _total}% confirmed)"
        )

    _overall = build_overall_source_distribution(_model_importances, _source_columns)
    if _overall:
        display(Markdown('### Overall Feature Origin (union across all models)'))
        _overall_df = native_pd.DataFrame(
            [{'source': s, 'feature_count': c} for s, c in _overall]
        )
        _pie = _px.pie(
            _overall_df, names='source', values='feature_count', hole=0.4,
            title='Share of trained features per source dataset',
        )
        display(_pie)
        display_table(_overall_df)


In [ ]:
# @cr:code name='save_diagnostics' id=8e138427
if not _skip_diagnostics and _diag_report:
    findings.metadata = findings.metadata or {}
    findings.metadata["model_diagnostics"] = _diag_report
    findings.save(FINDINGS_PATH)
    print("Diagnostics summary saved to findings metadata")


[//]: # (cr:doc name='9_diagnostics_to_business_separator' id=af136d24)

---
## Business Alignment

Now that we have assessed the model, align with business objectives.

[//]: # (cr:doc name='9_2_business_context' id=fa308f6e)
## 9.2 Business Context

Define the business context for this project.

In [ ]:
# @cr:user_code name='business_context' id=064d7a18
BUSINESS_CONTEXT = {
    "project_name": "Customer Churn Prediction",
    "business_objective": "Reduce customer churn by 20% through proactive retention campaigns",
    "stakeholders": ["Marketing Team", "Customer Success", "Data Science"],
    "timeline": "Q1 2025",
    "budget_constraints": "$50k for retention campaigns per month"
}

print("Business Context:")
for key, value in BUSINESS_CONTEXT.items():
    print(f"  {key}: {value}")

[//]: # (cr:doc name='9_3_success_metrics' id=a90ebf18)
## 9.3 Success Metrics

In [ ]:
# @cr:user_code name='success_metrics' id=b5d8f638
SUCCESS_METRICS = [
    {
        "Metric": "Model AUC",
        "Target": ">= 0.80",
        "Priority": "High",
        "Rationale": "Need strong discrimination to prioritize high-risk customers"
    },
    {
        "Metric": "Precision at 20%",
        "Target": ">= 0.60",
        "Priority": "High",
        "Rationale": "Limited budget means we can only target top 20% of predictions"
    },
    {
        "Metric": "Churn Rate Reduction",
        "Target": "20%",
        "Priority": "High",
        "Rationale": "Primary business objective"
    },
    {
        "Metric": "Model Latency",
        "Target": "< 100ms",
        "Priority": "Medium",
        "Rationale": "Required for real-time scoring"
    },
    {
        "Metric": "Fairness (Demographic Parity)",
        "Target": "Ratio >= 0.8",
        "Priority": "Medium",
        "Rationale": "Ensure equitable treatment across segments"
    }
]

metrics_df = native_pd.DataFrame(SUCCESS_METRICS)
print("Success Metrics:")
display_table(metrics_df)

[//]: # (cr:doc name='9_4_deployment_requirements' id=169f284c)
## 9.4 Deployment Requirements

In [ ]:
# @cr:user_code name='deployment_requirements' id=11c68fa9
DEPLOYMENT_REQUIREMENTS = {
    "scoring_mode": "Both batch and real-time",
    "batch_frequency": "Daily",
    "real_time_latency": "< 100ms p99",
    "infrastructure": "Databricks",
    "model_registry": "MLflow",
    "monitoring": "Required - drift detection and performance tracking",
    "retraining": "Monthly or on significant drift"
}

print("Deployment Requirements:")
for key, value in DEPLOYMENT_REQUIREMENTS.items():
    print(f"  {key}: {value}")

[//]: # (cr:doc name='9_5_data_constraints' id=12e610b1)
## 9.5 Data Constraints

In [ ]:
# @cr:user_code name='data_constraints' id=24109e0b
DATA_CONSTRAINTS = [
    {
        "Constraint": "PII Handling",
        "Requirement": "No direct PII in features (names, SSN, etc.)",
        "Status": "To verify"
    },
    {
        "Constraint": "Data Freshness",
        "Requirement": "Features must be available within 24 hours",
        "Status": "To verify"
    },
    {
        "Constraint": "Historical Depth",
        "Requirement": "Minimum 12 months of history for training",
        "Status": "To verify"
    },
    {
        "Constraint": "Protected Attributes",
        "Requirement": "Age, gender, race should not be direct features",
        "Status": "To verify"
    }
]

constraints_df = native_pd.DataFrame(DATA_CONSTRAINTS)
print("Data Constraints:")
display_table(constraints_df)

[//]: # (cr:doc name='9_6_intervention_strategy' id=fc454427)
## 9.6 Intervention Strategy

In [ ]:
# @cr:user_code name='interventions' id=67b11574
INTERVENTIONS = [
    {
        "Risk Level": "High (>0.8)",
        "Intervention": "Personal call from account manager",
        "Cost": "$50/customer",
        "Expected Effectiveness": "40% retention"
    },
    {
        "Risk Level": "Medium (0.5-0.8)",
        "Intervention": "Personalized email + discount offer",
        "Cost": "$10/customer",
        "Expected Effectiveness": "20% retention"
    },
    {
        "Risk Level": "Low (<0.5)",
        "Intervention": "Automated engagement email",
        "Cost": "$0.50/customer",
        "Expected Effectiveness": "5% retention"
    }
]

interventions_df = native_pd.DataFrame(INTERVENTIONS)
print("Intervention Strategy:")
display_table(interventions_df)

[//]: # (cr:doc name='9_7_save_business_context_to_findings' id=b7d3a943)
## 9.7 Save Business Context to Findings

In [ ]:
# @cr:code name='save_business_metadata' id=e62fbb91
findings.metadata = findings.metadata or {}
findings.metadata["business_context"] = BUSINESS_CONTEXT
findings.metadata["success_metrics"] = SUCCESS_METRICS
findings.metadata["deployment_requirements"] = DEPLOYMENT_REQUIREMENTS

findings.save(FINDINGS_PATH)
print(f"Business context saved to: {FINDINGS_PATH}")


In [ ]:
# @cr:code name='release_stage_memory' id=d339027c
from customer_retention.core.compat import release_stage_memory

release_stage_memory()

[//]: # (cr:doc name='next_steps' id=e960331a)
---

## Next Steps

Continue to **10_spec_generation.ipynb** to generate production specifications.

[//]: # (cr:doc name='section' id=de37c163)
> **Save Reminder:** Save this notebook (Ctrl+S / Cmd+S) before running the next one.
> The next notebook will automatically export this notebook's HTML documentation from the saved file.